In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('Athletes.xlsx')

In [3]:
df.head()

,Name,NOC,Discipline
0,AALERUD Katrine,Norway,Cycling Road
1,ABAD Nestor,Spain,Artistic Gymnastics
2,ABAGNALE Giovanni,Italy,Rowing
3,ABALDE Alberto,Spain,Basketball
4,ABALDE Tamara,Spain,Basketball


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11085 entries, 0 to 11084
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Name        11085 non-null  object
 1   NOC         11085 non-null  object
 2   Discipline  11085 non-null  object
dtypes: object(3)
memory usage: 259.9+ KB


In [5]:
df.describe()

,Name,NOC,Discipline
count,11085,11085,11085
unique,11062,206,46
top,PALTRINIERI Gregorio,United States of America,Athletics
freq,2,615,2068


In [11]:
#Obtain list of disciplines to map to new column Parent_Discipline
for i in df['Discipline'].sort_values().unique():
    print(i)

3x3 Basketball
Archery
Artistic Gymnastics
Artistic Swimming
Athletics
Badminton
Baseball/Softball
Basketball
Beach Volleyball
Boxing
Canoe Slalom
Canoe Sprint
Cycling BMX Freestyle
Cycling BMX Racing
Cycling Mountain Bike
Cycling Road
Cycling Track
Diving
Equestrian
Fencing
Football
Golf
Handball
Hockey
Judo
Karate
Marathon Swimming
Modern Pentathlon
Rhythmic Gymnastics
Rowing
Rugby Sevens
Sailing
Shooting
Skateboarding
Sport Climbing
Surfing
Swimming
Table Tennis
Taekwondo
Tennis
Trampoline Gymnastics
Triathlon
Volleyball
Water Polo
Weightlifting
Wrestling


In [13]:
#Creating Parent_Discipline to account for individual athletes competing in more than one discipline (1:many)
df['Parent_Discipline'] = df['Discipline'].map({
    'Artistic Swimming': 'Aquatics',
    'Diving': 'Aquatics',
    'Marathon Swimming': 'Aquatics',
    'Swimming': 'Aquatics',
    'Water Polo': 'Aquatics',
    '3x3 Basketball': 'Basketball',
    'Canoe Slalom': 'Canoeing',
    'Canoe Sprint': 'Canoeing',
    'Cycling BMX Freestyle': 'Cycling',
    'Cycling BMX Racing': 'Cycling',
    'Cycling Mountain Bike': 'Cycling',
    'Cycling Road': 'Cycling',
    'Cycling Track': 'Cycling',
    'Artistic Gymnastics': 'Gymnastics',
    'Rhythmic Gymnastics': 'Gymnastics',
    'Trampoline Gymnastics': 'Gymnastics',
    'Beach Volleyball': 'Volleyball',
})
df.head()

,Name,NOC,Discipline,Parent_Discipline
0,AALERUD Katrine,Norway,Cycling Road,Cycling
1,ABAD Nestor,Spain,Artistic Gymnastics,Gymnastics
2,ABAGNALE Giovanni,Italy,Rowing,NaN
3,ABALDE Alberto,Spain,Basketball,NaN
4,ABALDE Tamara,Spain,Basketball,NaN


In [14]:
#If Parent_Discipline is null at this point, fill NaN with the Discipline
df['Parent_Discipline'].fillna(df['Discipline'], inplace=True)
df.head()

,Name,NOC,Discipline,Parent_Discipline
0,AALERUD Katrine,Norway,Cycling Road,Cycling
1,ABAD Nestor,Spain,Artistic Gymnastics,Gymnastics
2,ABAGNALE Giovanni,Italy,Rowing,Rowing
3,ABALDE Alberto,Spain,Basketball,Basketball
4,ABALDE Tamara,Spain,Basketball,Basketball


In [27]:
df_parent = df.loc[:, ['Name', 'NOC', 'Parent_Discipline']]#[df.duplicated(keep=False)==False]
#df_parent[(df_parent.Parent_Discipline=='Aquatics') & (df_parent.Name=='van ROUWENDAAL Sharon')]

df_parent.drop_duplicates(inplace=True)

In [30]:
df_parent.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11077 entries, 0 to 11084
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               11077 non-null  object
 1   NOC                11077 non-null  object
 2   Parent_Discipline  11077 non-null  object
dtypes: object(3)
memory usage: 346.2+ KB


In [31]:
df_parent.describe()

,Name,NOC,Parent_Discipline
count,11077,11077,11077
unique,11062,206,33
top,KURBANOV Ruslan,United States of America,Athletics
freq,2,614,2068


In [34]:
df_parent[(df_parent.Parent_Discipline=='Aquatics') & (df_parent.Name=='van ROUWENDAAL Sharon')]

,Name,NOC,Parent_Discipline
10111,van ROUWENDAAL Sharon,Netherlands,Aquatics


In [32]:
#Find duplicated names and determine reason for them- keep all duplicated records
names_dupes = df_parent[df_parent.duplicated(subset='Name', keep=False)==True]
names_dupes#.count()

,Name,NOC,Parent_Discipline
319,ALVAREZ Jorge,Cuba,Shooting
320,ALVAREZ Jorge,Honduras,Football
1703,CHEN Yang,People's Republic of China,Athletics
1704,CHEN Yang,People's Republic of China,Hockey
3664,HALL James,Great Britain,Gymnastics
3665,HALL James,United States of America,Shooting
4829,KIM Hyunsoo,Republic of Korea,Baseball/Softball
4830,KIM Hyunsoo,Republic of Korea,Rugby Sevens
5106,KOVACS Zsofia,Hungary,Gymnastics
5107,KOVACS Zsofia,Hungary,Triathlon


In [33]:
names_dupes.to_csv('names_dupes.csv')

In [35]:
#check duplicates across all data
df_parent[df_parent.duplicated(keep=False)==True]

,Name,NOC,Parent_Discipline


In [36]:
#athletes in each NOC
df_parent.loc[:, ['Name', 'NOC']].groupby('NOC').count().sort_values(by='Name', ascending=False).head(10)

,Name
NOC,
United States of America,614
Japan,586
Australia,470
People's Republic of China,401
Germany,399
France,377
Canada,368
Great Britain,366
Italy,354


In [38]:
#These do not take into account *distinct* athletes. I.e. marathon swimmers also were swimmers
df.loc[:, ['Parent_Discipline', 'Discipline', 'Name']].groupby(['Parent_Discipline', 'Discipline']).count()

Name
Parent_Discipline Discipline                 
Aquatics          Artistic Swimming        98
                  Diving                  133
                  Marathon Swimming        49
                  Swimming                743
                  Water Polo              269
Archery           Archery                 122
Athletics         Athletics              2068
Badminton         Badminton               164
Baseball/Softball Baseball/Softball       220
Basketball        3x3 Basketball           62
                  Basketball              280
Boxing            Boxing                  270
Canoeing          Canoe Slalom             78
                  Canoe Sprint            236
Cycling           Cycling BMX Freestyle    19
                  Cycling BMX Racing       43
                  Cycling Mountain Bike    74
                  Cycling Road            190
                  Cycling Track           208
Equestrian        Equestrian              237
Fencing           Fencing                 249
Football          Football                567
Golf              Golf                    115
Gymnastics        Artistic Gymnastics     187
                  Rhythmic Gymnastics      95
                  Trampoline Gymnastics    31
Handball          Handball                343
Hockey            Hockey                  406
Judo              Judo                    373
Karate            Karate                   77
Modern Pentathlon Modern Pentathlon        69
Rowing            Rowing                  496
Rugby Sevens      Rugby Sevens            283
Sailing           Sailing                 336
Shooting          Shooting                342
Skateboarding     Skateboarding            77
Sport Climbing    Sport Climbing           37
Surfing           Surfing                  38
Table Tennis      Table Tennis            164
Taekwondo         Taekwondo               123
Tennis            Tennis                  178
Triathlon         Triathlon               106
Volleyball        Beach Volleyball         90
                  Volleyball              274
Weightlifting     Weightlifting           187
Wrestling         Wrestling               279

In [39]:
df_parent.loc[:, ['Parent_Discipline', 'Name']].groupby('Parent_Discipline').count()
#these take into account *distinct* athlete

,Name
Parent_Discipline,
Aquatics,1289
Archery,122
Athletics,2068
Badminton,164
Baseball/Softball,220
Basketball,342
Boxing,270
Canoeing,314
Cycling,530


In [49]:
aquatics = df[df.Parent_Discipline=='Aquatics']#.duplicated(subset='Name', keep=False)]
aquatics[aquatics.duplicated(subset='Name', keep=False)]

,Name,NOC,Discipline,Parent_Discipline
7395,PALTRINIERI Gregorio,Italy,Marathon Swimming,Aquatics
7396,PALTRINIERI Gregorio,Italy,Swimming,Aquatics
10111,van ROUWENDAAL Sharon,Netherlands,Marathon Swimming,Aquatics
10112,van ROUWENDAAL Sharon,Netherlands,Swimming,Aquatics
10505,WELLBROCK Florian,Germany,Marathon Swimming,Aquatics
10506,WELLBROCK Florian,Germany,Swimming,Aquatics


In [50]:
aquatics_parent = df_parent[df_parent.Parent_Discipline=='Aquatics']
aquatics_parent[aquatics_parent.duplicated(subset='Name', keep=False)]

,Name,NOC,Parent_Discipline


In [51]:
df_parent['Name'].count()

11077